In [ ]:
import pandas as pd

# 1. Load the data 
annotations_df = pd.read_csv('human_annotation.csv')
errors_df = pd.read_csv('kaycieresults/both_models_wrong_4class.csv')

# Clean the column names (Removes any hidden spaces or carriage returns)
annotations_df.columns = annotations_df.columns.str.strip()
errors_df.columns = errors_df.columns.str.strip()

# 2. Prevent duplicate column names (_x and _y)
# If errors_df already has 'notes', drop it so we can merge cleanly from the main source
existing_cols = [col for col in ['notes', 'dominant_emotion'] if col in errors_df.columns]
if existing_cols:
    errors_df = errors_df.drop(columns=existing_cols)

# 3. Merge to bring the fresh annotator notes into the error dataframe
analysis_df = pd.merge(errors_df, annotations_df[['track_name', 'notes', 'dominant_emotion']], 
                       on='track_name', 
                       how='left')

# 4. Define the linguistic categories from the proposal
categories = {
    'Slang / AAVE': ['slang', 'aave', 'vernacular'],
    'Polarity Inversion': ['polarity inversion', 'inversion', 'empowering', 'positive'],
    'Irony / Sarcasm': ['irony', 'sarcasm', 'sarcastic', 'ironic'],
    'Metaphor': ['metaphor', 'metaphorical', 'figurative'],
    'Narrative Voice': ['narrative', 'storytelling', 'persona']
}

# 5. Categorize the errors based on annotator notes
def categorize_error(note):
    if pd.isna(note):
        return "No Note"
    note_lower = str(note).lower()
    found_categories = []
    
    for category, keywords in categories.items():
        if any(keyword in note_lower for keyword in keywords):
            found_categories.append(category)
            
    return ", ".join(found_categories) if found_categories else "Other/Uncategorized"

analysis_df['error_category'] = analysis_df['notes'].apply(categorize_error)

# 6. Display the Systematic Pattern Summary
print("--- Error Breakdown by Linguistic Feature ---")
category_counts = analysis_df['error_category'].value_counts()
print(category_counts)
print("\n")

# 7. Extract specific examples for the final paper write-up
print("--- Sample Cases: Polarity Inversion ---")
polarity_cases = analysis_df[analysis_df['error_category'].str.contains('Polarity Inversion', na=False)]

for index, row in polarity_cases.head(3).iterrows():
    print(f"Track: {row['track_name']}")
    print(f"Annotator Note: {row['notes']}")
    print("-" * 40)

--- Error Breakdown by Linguistic Feature ---
error_category
Narrative Voice                  10
Polarity Inversion                7
Other/Uncategorized               5
Slang / AAVE                      5
No Note                           4
Irony / Sarcasm                   2
Slang / AAVE, Narrative Voice     1
Name: count, dtype: int64


--- Sample Cases: Polarity Inversion ---
Track: Codeine Crazy
Annotator Note: addiction framed positively (luxury + success); polarity inversion (“lovely” while describing substance abuse)

 models likely overpredicts positive
----------------------------------------
Track: KOD
Annotator Note: drug culture critique vs glorification; negative themes expressed with confident tone; likely misclassified as positive
----------------------------------------
Track: Plain Jane REMIX (feat. Nicki Minaj)
Annotator Note: likely polarity inversion issues
----------------------------------------
